## Retrieval Augmented Generation (RAG)

### Instalacja bibliotek

In [1]:
# Zainstaluj bezpośrednie zależności używane w tym notebooku.
%pip install -qU langchain-openai langchain-community langchain-text-splitters faiss-cpu python-dotenv
# Google Colab: dodaj OPENAI_API_KEY w panelu Secrets (🔑) i zezwól notatnikowi na dostęp.
# Odkomentuj te linie, aby wczytać klucz z Colab Secrets do bieżącej sesji:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# Lokalnie umieść OPENAI_API_KEY=... w pliku .env; nie commituj klucza.

Note: you may need to restart the kernel to use updated packages.


### Budowa VectorStore (FAISS) i Retrievera

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

load_dotenv()

docs = [
    "LangChain to framework do pracy z LLM.",
    "RAG łączy retrieval kontekstu z generacją odpowiedzi.",
    "FAISS to biblioteka do przechowywania i wyszukiwania embeddingów.",
    "Retriever służy do wyszukiwania najbardziej podobnych dokumentów do zapytania użytkownika. Retriever może zwrócić różną liczbę pasujących dokumentów określona w parametrze k. Retriever wykorzystuje różne algorytmy podobieństwa tekstów, np. dopasowanie kosinusowe, odległość euklidesowa, MMR."
]

splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
split = splitter.create_documents(docs)

print(f"Number of chunks: {len(split)}")

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(split, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

query = "Po co używa się retrievera?"

context = retriever.invoke(query)
print("Znaleziony fragment:")
for i, c in enumerate(context, 1):
    print(f"{i}.", c.page_content)

/tmp/ipykernel_377802/2090219182.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Number of chunks: 7
Znaleziony fragment:
1. Retriever służy do wyszukiwania najbardziej podobnych dokumentów do zapytania użytkownika. Retriever
2. Retriever może zwrócić różną liczbę pasujących dokumentów określona w parametrze k. Retriever


### Prosty łańcuch RAG (prompt + kontekst + LLM)

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-6-luna")

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Udziel precyzyjnej odpowiedzi wyłącznie na podstawie KONTEKSTU. Jeśli brak danych — powiedz, że nie wiesz."),
    ("system", "KONTEKST:\n{context}"),
    ("user", "{question}")
])

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("Czym jest FAISS i do czego służy?"))

FAISS to biblioteka do przechowywania i wyszukiwania embeddingów. Służy do znajdowania dokumentów podobnych do zapytania użytkownika.


### Przykład RAG - cały program

In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Model
llm = ChatOpenAI(model="gpt-6-luna")

# Dokumenty źródłowe
docs = [
    "LangChain to framework do pracy z LLM.",
    "RAG łączy dopasowanie kontekstu z generacją odpowiedzi.",
    "FAISS to biblioteka do przechowywania i wyszukiwania embeddingów."
]

# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
splits = splitter.create_documents(docs)

# Embeddings + vector store
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Prompt RAG
prompt = ChatPromptTemplate.from_messages([
    ("system", "Odpowiadaj tylko na podstawie kontekstu:\n{context}"),
    ("user", "{question}")
])

# Pipeline
rag_chain = (
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"]
    }
    | prompt
    | llm
    | StrOutputParser()
)


print(rag_chain.invoke({"question": "Co to jest FAISS?"}))


FAISS to biblioteka do przechowywania.


### RAG z pętlą i ewaluacją

In [5]:
from langchain_core.prompts import ChatPromptTemplate

eval_prompt = ChatPromptTemplate.from_messages([
    ("system", "Oceń odpowiedź."),
    ("user", "Pytanie: {question}\nOdpowiedź: {answer}\nCzy odpowiedź jest poprawna? Odpowiedz yes/no.")
])

def rag_with_eval(question, max_retries):
    for attempt in range(max_retries):
        context = retriever.invoke(question)
        answer = (prompt | llm | StrOutputParser()).invoke({"context": context, "question": question})
        eval_result = (eval_prompt | llm | StrOutputParser()).invoke({"question": question, "answer": answer})
        print(f"Wynik ewaluacji {eval_result}")
        if "yes" in eval_result.lower():
            return f"✅ Odpowiedź zaakceptowana:\n{answer}"
        print(f"❌ Odpowiedź: {answer}\n odrzucona, ponawiam próbę...")
    return "Nie udało się uzyskać poprawnej odpowiedzi."

print(rag_with_eval("Co to jest RAG?", max_retries=3))


Wynik ewaluacji yes
✅ Odpowiedź zaakceptowana:
RAG łączy dopasowanie kontekstu z generacją odpowiedzi.
